Initialization

In [1]:
import pandas as pd
import numpy as np
import cv2
import os

# Set working directory

os.chdir(r"C:\Users\turab\OneDrive\Desktop\Wildfire_Railway")

Decode telemetry

In [4]:
def decode_file(txt_path, output_csv_path):
    """
    Decodes the raw .txt file into a structured .csv file.

    Parameters:
        txt_path (str): Path to the raw telemetry .txt file.
        output_csv_path (str): Path to save the decoded telemetry .csv file.

    Returns:
        pd.DataFrame: Decoded telemetry data.
    """
    try:
        with open(txt_path, 'r') as file:
            raw_data = file.readlines()

        # Extract telemetry data 
        
        decoded_data = []
        for line in raw_data:
            parts = line.strip().split(',')
            if len(parts) > 5: 
                decoded_data.append({
                    'timestamp': parts[0],
                    'latitude': parts[1],
                    'longitude': parts[2],
                    'altitude': parts[3],
                    'other_field': parts[4]
                })

        df = pd.DataFrame(decoded_data)
        df.to_csv(output_csv_path, index=False)
        print(f"Decoded telemetry saved to {output_csv_path}")
        return df

    except Exception as e:
        print(f"Error decoding file: {e}")
        return None

Interpolation function

In [5]:
def interpolate_telemetry(telemetry_data, frame_timestamps):
    """
    Interpolates telemetry data to align with video frame timestamps.

    Parameters:
        telemetry_data (pd.DataFrame): Telemetry data with 'timestamp' and other fields.
        frame_timestamps (list or np.array): List of frame timestamps (in milliseconds).

    Returns:
        pd.DataFrame: Interpolated telemetry data aligned with frame timestamps.
    """
    telemetry_data['timestamp'] = pd.to_datetime(telemetry_data['timestamp'])
    telemetry_data['timestamp_ms'] = telemetry_data['timestamp'].astype(np.int64) // 10**6

    interpolated = pd.DataFrame({'frame_timestamp': frame_timestamps})
    for col in telemetry_data.columns:
        if col not in ['timestamp', 'timestamp_ms']:
            interpolated[col] = np.interp(
                frame_timestamps,
                telemetry_data['timestamp_ms'],
                telemetry_data[col]
            )

    return interpolated

overlay text on frame

In [6]:
def overlay_text_on_frame(frame, telemetry_row):
    """
    Overlays telemetry data as text on a video frame with a black background for readability.

    Parameters:
        frame (numpy.ndarray): The video frame.
        telemetry_row (pd.Series): A row of telemetry data containing the fields to overlay.

    Returns:
        numpy.ndarray: The video frame with overlaid text.
    """
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    thickness = 2
    y_offset = 30  

    # Define the data to overlay 

    overlay_data = {
        "Time": telemetry_row.get('timestamp', 'N/A'),
        "Latitude": telemetry_row.get('latitude', 'N/A'),
        "Longitude": telemetry_row.get('longitude', 'N/A'),
        "Altitude": telemetry_row.get('altitude', 'N/A')
    }

    for key, value in overlay_data.items():
        text = f"{key}: {value}"
        text_size = cv2.getTextSize(text, font, font_scale, thickness)[0]

        # Black rectangle as background for the text

        cv2.rectangle(
            frame, 
            (10, y_offset - 20), 
            (10 + text_size[0] + 10, y_offset + 10), 
            (0, 0, 0), 
            -1
        )

        # Overlay the text

        cv2.putText(
            frame, 
            text, 
            (10, y_offset), 
            font, 
            font_scale, 
            (255, 255, 255),  
            thickness, 
            cv2.LINE_AA
        )

        y_offset += 30  

    return frame

Process video with overlay

In [7]:
def process_video_with_overlay(video_path, frame_data, output_video_path, duration_minutes=2, frame_skip=3):
    """
    Processes the video by overlaying telemetry data on frames and saving the output.

    Parameters:
        video_path (str): Path to the input video file.
        frame_data (pd.DataFrame): Telemetry data synchronized with frames.
        output_video_path (str): Path to save the processed video.
        duration_minutes (int): Duration of video to process in minutes.
        frame_skip (int): Number of frames to skip for text updates.
    """
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    max_frames = int(fps * 60 * duration_minutes)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    last_data_index = 0  

    for i in range(min(max_frames, frame_count)):
        ret, frame = cap.read()
        if not ret:
            break

        # Update telemetry text every 'frame_skip' frames

        if i % frame_skip == 0 and last_data_index < len(frame_data):
            telemetry_row = frame_data.iloc[last_data_index]
            last_data_index += 1

        # Overlay the last telemetry row on the current frame

        frame = overlay_text_on_frame(frame, telemetry_row)

        out.write(frame)

        if i % 100 == 0:
            print(f"Processed {i} frames...")

    cap.release()
    out.release()
    print(f"Processed video saved at: {output_video_path}")

Example usage

In [ ]:
# Raw .txt file decoding

txt_path = "path/to/raw_file.txt"  
output_csv_path = "path/to/decoded_data.csv"  

# Decode the raw telemetry file

decoded_data = decode_file(txt_path, output_csv_path)

# Display the first few rows of the decoded data

if decoded_data is not None:
    print(decoded_data.head())

In [ ]:
# Interpolating telemetry data

# Example frame timestamps (assuming 30 fps for 60 seconds)
frame_timestamps = np.linspace(0, 60000, 1800)  

# Interpolate telemetry data to match frame timestamps

if decoded_data is not None:
    interpolated_data = interpolate_telemetry(decoded_data, frame_timestamps)

    # Display the first few rows of the interpolated data

    print(interpolated_data.head())

In [ ]:
# Processing video with telemetry overlay

video_path = "path/to/video.mp4"  
output_video_path = "path/to/output_video.mp4"  

# Process the video and overlay telemetry data
if 'interpolated_data' in locals():
    process_video_with_overlay(video_path, interpolated_data, output_video_path, duration_minutes=2, frame_skip=3)
else:
    print("Interpolated data is missing. Please run the interpolation step.")